# 01.5 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/05-pandas-clean/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

raw = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
print(raw.shape)
raw.head()

(541910, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Exercise 1

In [2]:
valid = raw[~raw["Invoice"].astype(str).str.startswith("C")]
valid = valid[(valid["Quantity"] > 0) & (valid["Price"] > 0)]
rev = valid["Quantity"] * valid["Price"]
guest = valid["Customer ID"].isna()
print(f"guest share of rows:    {guest.mean():.1%}")
print(f"guest share of revenue: {rev[guest].sum() / rev.sum():.1%}")

guest share of rows:    24.9%
guest share of revenue: 16.5%


Guests are a quarter of rows but a smaller share of revenue (smaller baskets). A *finance* report must keep them: the money is real. A *customer* model must drop them: there is no customer to model. Same data, two cleanings.

## Exercise 2

In [3]:
names_per_code = raw.groupby("StockCode")["Description"].nunique()
multi = names_per_code[names_per_code > 1]
print("codes with >1 description:", len(multi))
raw[raw["StockCode"].isin(multi.index[:3])].groupby("StockCode")["Description"].unique()

codes with >1 description: 650


StockCode
10080          [GROOVY CACTUS INFLATABLE, nan, check]
10133         [COLOURING PENCILS BROWN TUBE, damaged]
16008    [SMALL FOLDING SCISSOR(POINTED EDGE), check]
Name: Description, dtype: object

In [4]:
# rule: use the most frequent non-missing description for each code
canonical = (raw.dropna(subset=["Description"])
                .groupby("StockCode")["Description"]
                .agg(lambda s: s.value_counts().index[0]))
fixed = raw.copy()
fixed["Description"] = fixed["StockCode"].map(canonical)
print("codes with >1 description after fix:", (fixed.groupby("StockCode")["Description"].nunique() > 1).sum())

codes with >1 description after fix: 0


## Exercise 3

In [5]:
flagged = raw.copy()
flagged["is_cancelled"] = flagged["Invoice"].astype(str).str.startswith("C")
flagged["gross"] = flagged["Quantity"].abs() * flagged["Price"]
nov = flagged[flagged["InvoiceDate"].dt.to_period("M") == "2011-11"]
print(f"November 2011 cancelled value: £{nov.loc[nov['is_cancelled'], 'gross'].sum():,.0f}")
print(f"as share of November sales:     {nov.loc[nov['is_cancelled'], 'gross'].sum() / nov.loc[~nov['is_cancelled'], 'gross'].sum():.1%}")

November 2011 cancelled value: £47,740
as share of November sales:     3.2%


## Exercise 4

In [6]:
before = raw["Country"].memory_usage(deep=True)
after = raw["Country"].astype("category").memory_usage(deep=True)
print(f"before: {before/1e6:.1f} MB   after: {after/1e6:.2f} MB   ({before/after:.0f}× smaller)")

before: 11.6 MB   after: 0.54 MB   (21× smaller)


A column with 38 distinct values repeated 541,910 times stores each value once as a category and an integer code per row. Use `category` for low-cardinality text columns; it is also what one-hot encoding (Module 04) expects.